In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import logging

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from mamba_ssm import Mamba

# ==========================================
# Global configuration
# ==========================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for visualization")
device = torch.device("cuda")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

TARGET_LEN = 6000
INPUT_CHANNEL = 3
FS = 100.0
PICK_TOLERANCE = 50
PICK_THRESHOLD_P = 0.02
PICK_THRESHOLD_S = 0.08

BATCH_SIZE = 32
OUTPUT_DIR = "./outputs_selected_waveforms_optimized"   # new output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_PATH = "./outputs/model_best.pt"                  # trained model
REAL_NOISE_PATH = "./stead_real_noise_50000.npy"
METADATA_PATH = "./metadata.csv"

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Plot style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.dpi'] = 300
FIXED_YLIM = [-15, 15]

# ==========================================
# Helper functions (identical to training)
# ==========================================
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data, axis=-1)
    return y

def generate_bandpass_noise(length, fs, band, order=4):
    lowcut, highcut = band
    white_noise = np.random.randn(length)
    band_noise = bandpass_filter(white_noise, lowcut, highcut, fs, order)
    return band_noise

def add_noise_with_snr(signal, snr_db, fs, noise_band):
    signal_noisy = np.zeros_like(signal)
    for ch in range(signal.shape[1]):
        sig = signal[:, ch]
        sig_power = np.mean(sig ** 2)
        noise_power = sig_power / (10 ** (snr_db / 10))
        noise = generate_bandpass_noise(len(sig), fs, noise_band)
        noise_scaled = noise * np.sqrt(noise_power / (np.mean(noise ** 2) + 1e-10))
        signal_noisy[:, ch] = sig + noise_scaled
    return signal_noisy

def shift_waveform_and_label(waveform, label, max_shift=500):
    shift = np.random.randint(-max_shift, max_shift)
    shifted_wave = np.zeros_like(waveform)
    shifted_label = np.zeros_like(label)
    if shift > 0:
        shifted_wave[shift:, :] = waveform[:-shift, :]
        shifted_label[shift:, :] = label[:-shift, :]
    elif shift < 0:
        shifted_wave[:shift, :] = waveform[-shift:, :]
        shifted_label[:shift, :] = label[-shift:, :]
    else:
        shifted_wave = waveform
        shifted_label = label
    return shifted_wave, shifted_label, shift

def normalize_single_sample(x):
    x_normalized = np.zeros_like(x, dtype=np.float32)
    for ch in range(INPUT_CHANNEL):
        mean = np.mean(x[:, ch])
        std = np.std(x[:, ch])
        if std < 1e-6:
            std = 1.0
        x_normalized[:, ch] = (x[:, ch] - mean) / std
    return x_normalized

def generate_triangle_label(length, peak_idx, width=40):
    label = np.zeros(length, dtype=np.float32)
    half_width = width // 2
    start = max(0, peak_idx - half_width)
    end = min(length, peak_idx + half_width)
    rise_len = peak_idx - start
    if rise_len > 0:
        label[start:peak_idx] = np.linspace(0, 1, rise_len)
    label[peak_idx] = 1.0
    fall_len = end - peak_idx
    if fall_len > 0:
        label[peak_idx+1:end] = np.linspace(1, 0, fall_len-1)
    return label

def generate_3channel_labels(y_2channel, width_p=20, width_s=40):
    n_samples, n_timesteps = y_2channel.shape[0], y_2channel.shape[1]
    y_3channel = np.zeros((n_samples, n_timesteps, 3), dtype=np.float32)
    for i in range(n_samples):
        p_idx = np.argmax(y_2channel[i, :, 0])
        s_idx = np.argmax(y_2channel[i, :, 1])
        y_3channel[i, :, 0] = generate_triangle_label(n_timesteps, p_idx, width=width_p)
        y_3channel[i, :, 1] = generate_triangle_label(n_timesteps, s_idx, width=width_s)
    y_3channel[..., 2] = 1.0 - y_3channel[..., 0] - y_3channel[..., 1]
    return np.clip(y_3channel, 0.0, 1.0)

def calculate_snr(waveform, p_idx, s_idx):
    signal_start = max(0, p_idx - 100)
    signal_end = min(len(waveform), s_idx + 200)
    noise_start = max(0, signal_start - 500)
    noise_end = signal_start
    if noise_end <= noise_start:
        return 0.0
    signal_power = np.mean(waveform[signal_start:signal_end, 2] ** 2)
    noise_power = np.mean(waveform[noise_start:noise_end, 2] ** 2)
    if noise_power < 1e-10:
        return 100.0
    return 10 * np.log10(signal_power / noise_power)

# ==========================================
# Optimized model (num_heads=4, kernel=13, no redundant pre_norm)
# ==========================================
class StochasticDepthDropout(nn.Module):
    def __init__(self, drop_rate):
        super().__init__()
        self.drop_rate = drop_rate

    def forward(self, layer_output, residual):
        if self.training:
            keep_prob = 1 - self.drop_rate
            if torch.rand(1).item() < keep_prob:
                return residual + (layer_output / keep_prob)
            else:
                return residual
        else:
            return residual + layer_output

class DilatedConvBlock(nn.Module):
    """Two dilated conv layers with residual connection."""
    def __init__(self, in_channels, filters, kernel_size=13, dilation_rates=[1, 2], dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filters, kernel_size,
                               dilation=dilation_rates[0], padding='same')
        self.norm1 = nn.LayerNorm(filters)
        self.act1 = nn.ReLU()
        self.conv2 = nn.Conv1d(filters, filters, kernel_size,
                               dilation=dilation_rates[1], padding='same')
        self.norm2 = nn.LayerNorm(filters)
        self.act2 = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.residual_conv = nn.Conv1d(in_channels, filters, 1) if in_channels != filters else nn.Identity()

    def forward(self, x):
        residual = self.residual_conv(x)
        x = self.conv1(x)
        x = self.norm1(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act1(x)
        x = self.conv2(x)
        x = self.norm2(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act2(x)
        x = self.dropout(x)
        return x + residual

class DualMambaAttentionBlock(nn.Module):
    """Multi-head attention + bidirectional Mamba (no redundant pre_norm)."""
    def __init__(self, channels, num_heads=4, dropout=0.1, drop_rate=0.0):
        super().__init__()
        self.channels = channels
        self.pre_conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=13, padding='same'),
            nn.ReLU()
        )
        self.mha_norm = nn.LayerNorm(channels)
        self.mha = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.sdd_mha = StochasticDepthDropout(drop_rate)
        self.mamba_fwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.mamba_bwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.fusion = nn.Linear(2 * channels, channels)
        self.mamba_norm = nn.LayerNorm(channels)
        self.sdd_mamba = StochasticDepthDropout(drop_rate)

    def forward(self, x):
        residual_att = x
        x = self.pre_conv(x)
        x = x.permute(0, 2, 1)
        x_norm = self.mha_norm(x)
        att_out, _ = self.mha(x_norm, x_norm, x_norm)
        att_out = att_out.permute(0, 2, 1)
        x = self.sdd_mha(att_out, residual_att)

        residual_mamba = x
        temp_x = x.permute(0, 2, 1)
        out_fwd = self.mamba_fwd(temp_x)
        temp_x_rev = torch.flip(temp_x, dims=[1])
        out_bwd = self.mamba_bwd(temp_x_rev)
        out_bwd = torch.flip(out_bwd, dims=[1])
        out_combined = torch.cat([out_fwd, out_bwd], dim=-1)
        temp_x = self.fusion(out_combined)
        temp_x = self.mamba_norm(temp_x)
        temp_x = temp_x.permute(0, 2, 1)
        x = self.sdd_mamba(temp_x, residual_mamba)
        return x

class EncoderBlock(nn.Module):
    """Dilated conv block + downsampling."""
    def __init__(self, in_channels, out_channels, dilation_rates, dropout=0.1):
        super().__init__()
        self.dilated = DilatedConvBlock(in_channels, out_channels, 13, dilation_rates, dropout)
        self.down = nn.Conv1d(out_channels, out_channels, 13, stride=2, padding=6)

    def forward(self, x):
        skip = self.dilated(x)
        down = F.relu(self.down(skip))
        return skip, down

class DecoderBlock(nn.Module):
    """Upsampling + skip connection + two conv layers."""
    def __init__(self, in_channels, out_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose1d(in_channels, out_channels, kernel_size=13,
                                     stride=2, padding=6, output_padding=1)
        conv_in = out_channels + skip_channels
        self.conv = nn.Sequential(
            nn.Conv1d(conv_in, out_channels, 13, padding=6),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, 13, padding=6),
            nn.ReLU()
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = F.relu(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

class PhaseNet_Custom_DoubleMamba(nn.Module):
    """U-Net with dual Mamba bottleneck, num_heads=4."""
    def __init__(self, num_heads=4):
        super().__init__()
        self.enc1 = EncoderBlock(INPUT_CHANNEL, 16, [1, 2])
        self.enc2 = EncoderBlock(16, 32, [3, 4])
        self.enc3 = EncoderBlock(32, 64, [5, 6])
        self.enc4 = EncoderBlock(64, 128, [7, 8])

        self.dma1 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.0)
        self.dma2 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.1)

        self.dec4 = DecoderBlock(128, 128, 128)
        self.dec3 = DecoderBlock(128, 64, 64)
        self.dec2 = DecoderBlock(64, 32, 32)
        self.dec1 = DecoderBlock(32, 16, 16)

        self.final_conv = nn.Conv1d(16, 3, 1, padding='same')
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.permute(0, 2, 1)            # (B, C, T)

        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        x = self.dma1(x)
        x = self.dma2(x)

        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        x = self.final_conv(x)
        x = x.permute(0, 2, 1)            # (B, T, C)
        x = torch.softmax(x, dim=-1)
        return x

# ==========================================
# Inference dataset (normalization only)
# ==========================================
class SimpleInferenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        x_raw = self.X[idx]
        x_norm = normalize_single_sample(x_raw)
        return torch.tensor(x_norm, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

# ==========================================
# Waveform plotting (fixed x-axis ticks on probability subplot)
# ==========================================
def plot_single_waveform(sample_idx, raw_wave, true_label, pred_probs, magnitude, snr_val,
                         p_error, s_error, save_path):
    t = np.arange(TARGET_LEN)
    comp_names = ['E', 'N', 'Z']
    comp_color = '#1f77b4'
    color_p_true = '#cccc00'
    color_s_true = '#ff0000'
    color_p_pred = '#00aa00'
    color_s_pred = '#aa00aa'
    color_noise = '#666666'

    tp_true = np.argmax(true_label[:, 0])
    ts_true = np.argmax(true_label[:, 1])
    tp_pred = np.argmax(pred_probs[:, 0])
    ts_pred = np.argmax(pred_probs[:, 1])

    fig = plt.figure(figsize=(12, 11), dpi=300)
    gs = fig.add_gridspec(5, 1, height_ratios=[0.8, 1, 1, 1, 1.2], hspace=0.12)

    x_lim_start = max(0, tp_true - 1000)
    x_lim_end = min(TARGET_LEN, ts_true + 1000)

    ax_title = fig.add_subplot(gs[0, 0])
    ax_title.axis('off')
    ax_title.text(0.5, 0.5, f'M={magnitude:.1f}, SNR={snr_val:.1f} dB',
                  ha='center', va='center', fontsize=14, fontweight='bold')

    axes_wave = []
    norm_wave = normalize_single_sample(raw_wave)
    for i in range(3):
        ax = fig.add_subplot(gs[i+1, 0])
        ax.plot(t, norm_wave[:, i], color=comp_color, linewidth=0.8, label=comp_names[i])
        ax.axvline(tp_true, color=color_p_true, linestyle='-', linewidth=1.5, label='P Manual')
        ax.axvline(ts_true, color=color_s_true, linestyle='-', linewidth=1.5, label='S Manual')
        ax.set_ylabel('Amplitude', fontsize=11)
        ax.set_ylim(FIXED_YLIM)
        ax.set_yticks([-15, 0, 15])
        ax.set_yticklabels(['-1', '0', '1'])
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(alpha=0.2)
        ax.set_xlim([x_lim_start, x_lim_end])
        ax.set_xticks([])          # hide x-ticks on waveform rows
        axes_wave.append(ax)

    # Probability subplot WITHOUT shared x-axis (independent ticks)
    ax_prob = fig.add_subplot(gs[4, 0])
    ax_prob.plot(t, pred_probs[:, 0], color=color_p_pred, linestyle='--', linewidth=1.2, label='P Pick Probability')
    ax_prob.plot(t, pred_probs[:, 1], color=color_s_pred, linestyle='--', linewidth=1.2, label='S Pick Probability')
    ax_prob.plot(t, pred_probs[:, 2], color=color_noise, linestyle='--', linewidth=1.2, label='Noise Probability')
    err_str = f'P error={p_error:.2f}s  S error={s_error:.2f}s'
    ax_prob.text(0.3, 0.6, err_str, transform=ax_prob.transAxes, fontsize=12, fontweight='bold')
    ax_prob.set_ylim([0, 1.05])
    ax_prob.set_ylabel('Probability', fontsize=11)
    ax_prob.set_xlabel('Samples', fontsize=11)
    ax_prob.legend(fontsize=9, loc='lower right', ncol=1)
    ax_prob.grid(alpha=0.2)
    ax_prob.set_xlim([x_lim_start, x_lim_end])
    # Show x-axis tick labels
    ax_prob.tick_params(labelbottom=True)
    ax_prob.xaxis.set_major_locator(plt.MaxNLocator(6))
    ax_prob.xaxis.set_major_formatter(plt.FormatStrFormatter('%d'))

    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# ==========================================
# Load test events
# ==========================================
def load_test_events():
    logger.info("Loading STEAD test set earthquake events...")
    X_orig = np.load("chunk2_stead_X.npy").astype(np.float32)
    y_2ch = np.load("chunk2_stead_y.npy").astype(np.float32)
    y_3ch = generate_3channel_labels(y_2ch, width_p=20, width_s=40)

    metadata = pd.read_csv(METADATA_PATH, low_memory=False)
    magnitudes = metadata['source_magnitude'].values.astype(np.float32)
    assert len(magnitudes) == len(X_orig), "Magnitude count mismatch"

    # Fixed split (same as training)
    indices = np.arange(len(X_orig))
    np.random.seed(SEED)
    np.random.shuffle(indices)
    n_total_eq = len(X_orig)
    n_train_eq = int(n_total_eq * 0.7)
    n_val_eq = int(n_total_eq * 0.2)
    test_eq_idx = indices[n_train_eq + n_val_eq:]

    X_test_eq = X_orig[test_eq_idx]
    y_test_eq = y_3ch[test_eq_idx]
    test_magnitudes = magnitudes[test_eq_idx]

    logger.info(f"Test set earthquakes: {len(X_test_eq)}")
    return X_test_eq, y_test_eq, test_magnitudes

# ==========================================
# Select samples by criteria
# ==========================================
def select_samples_by_criteria(X_test_eq, y_test_eq, test_magnitudes):
    """
    Categories:
      - large magnitude: 4.0 <= mag <= 5.0
      - small magnitude: 0.1 <= mag <= 1.0
      - low SNR: 5 <= SNR <= 10 dB
    Select up to 2 samples per category (random, independent of global seed)
    """
    snr_list = []
    for i in range(len(X_test_eq)):
        p_idx = np.argmax(y_test_eq[i, :, 0])
        s_idx = np.argmax(y_test_eq[i, :, 1])
        snr = calculate_snr(X_test_eq[i], p_idx, s_idx)
        snr_list.append(snr)
    snr_list = np.array(snr_list)

    candidates = {'large_mag': [], 'small_mag': [], 'low_snr': []}
    for i, mag in enumerate(test_magnitudes):
        if 4.0 <= mag <= 5.0:
            candidates['large_mag'].append(i)
        if 0.1 <= mag <= 1.0:
            candidates['small_mag'].append(i)
    for i, snr in enumerate(snr_list):
        if 5.0 <= snr <= 10.0:
            candidates['low_snr'].append(i)

    rng = np.random.default_rng()   # independent random generator
    selected = {}
    for key, idx_list in candidates.items():
        n_available = len(idx_list)
        if n_available == 0:
            logger.warning(f"Category '{key}' has no samples.")
            selected[key] = []
        else:
            n_select = min(2, n_available)
            chosen = rng.choice(idx_list, size=n_select, replace=False)
            samples = []
            for idx in chosen:
                samples.append({
                    'idx': idx,
                    'waveform': X_test_eq[idx],
                    'label': y_test_eq[idx],
                    'magnitude': test_magnitudes[idx],
                    'snr': snr_list[idx]
                })
            selected[key] = samples
            logger.info(f"Category '{key}': {n_available} candidates, picked {len(samples)}")
    return selected

# ==========================================
# Main
# ==========================================
def main():
    logger.info("="*80)
    logger.info("  Generate custom waveforms: large mag (4-5), small mag (0.1-1), low SNR (5-10 dB) x2 each")
    logger.info("="*80)

    X_test_eq, y_test_eq, test_magnitudes = load_test_events()
    selected_samples = select_samples_by_criteria(X_test_eq, y_test_eq, test_magnitudes)

    logger.info(f"Loading optimized model: {MODEL_PATH}")
    if not os.path.exists(MODEL_PATH):
        logger.error(f"Model not found: {MODEL_PATH}")
        return
    model = PhaseNet_Custom_DoubleMamba(num_heads=4).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    model.eval()

    for cat, samples in selected_samples.items():
        for i, samp in enumerate(samples):
            idx = samp['idx']
            raw_wave = samp['waveform']
            true_label = samp['label']
            mag = samp['magnitude']
            snr = samp['snr']

            norm_wave = normalize_single_sample(raw_wave)
            inp = torch.tensor(norm_wave, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = model(inp).cpu().numpy()[0]

            tp_true = np.argmax(true_label[:, 0])
            ts_true = np.argmax(true_label[:, 1])
            tp_pred = np.argmax(pred[:, 0])
            ts_pred = np.argmax(pred[:, 1])
            p_error = (tp_pred - tp_true) / FS
            s_error = (ts_pred - ts_true) / FS

            prefix = f"{cat}_{i+1}_idx{idx}"
            save_path = os.path.join(OUTPUT_DIR, f"{prefix}_waveform.png")
            plot_single_waveform(idx, raw_wave, true_label, pred, mag, snr,
                                 p_error, s_error, save_path)

    logger.info(f"\nAll waveform figures saved to: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

2026-06-05 00:59:31,244 - ================================================================================
2026-06-05 00:59:31,245 -   Generate custom waveforms: large mag (4-5), small mag (0.1-1), low SNR (5-10 dB) x2 each
2026-06-05 00:59:31,246 - ================================================================================
2026-06-05 00:59:31,246 - Loading STEAD test set earthquake events...
2026-06-05 00:59:59,756 - Test set earthquakes: 20000
2026-06-05 01:00:00,499 - Category 'large_mag': 310 candidates, picked 2
2026-06-05 01:00:00,501 - Category 'small_mag': 8554 candidates, picked 2
2026-06-05 01:00:00,502 - Category 'low_snr': 1557 candidates, picked 2
2026-06-05 01:00:00,502 - Loading optimized model: ./outputs/model_best.pt
2026-06-05 01:00:03,379 - 
All waveform figures saved to: ./outputs_selected_waveforms_optimized
